# Stress Test: High-Capacity Memory Filtering
Test selective storage at 30-50% capacity with repetitive rejection.
Run on Kaggle GPU (T4) with this notebook.

In [ ]:
# Install dependencies
!pip install -q unsloth transformers accelerate bitsandbytes torch

In [ ]:
import os
import sys
import subprocess
import shutil
from pathlib import Path

REPO_URL = os.getenv("AL_NISYAN_REPO_URL", "https://github.com/faresrafat3/al-nisyan.git").strip()
working_repo = Path("/kaggle/working/al-nisyan")
input_root = Path("/kaggle/input")
cwd_repo = Path.cwd()

def is_repo_root(path: Path) -> bool:
    return (path / "src").exists() and (path / "experiments").exists()

candidates = [working_repo, cwd_repo]
if input_root.exists():
    candidates.extend([p for p in input_root.iterdir() if p.is_dir()])
project_root = None
for candidate in candidates:
    if is_repo_root(candidate):
        project_root = candidate
        break
    for nested in candidate.glob("**/*") if candidate.exists() else []:
        if nested.is_dir() and is_repo_root(nested):
            project_root = nested
            break
    if project_root is not None:
        break

if project_root is not None and project_root != working_repo:
    if working_repo.exists():
        shutil.rmtree(working_repo)
    print(f"Copying project to writable path: {working_repo}")
    shutil.copytree(project_root, working_repo)
    project_root = working_repo
elif project_root is None:
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, str(working_repo)], check=True)
    project_root = working_repo

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Ready to run stress test")

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# Run the stress test
from experiments.capacity_stress_test import stress_test

print("Starting stress test (50 interactions to fill memory 30%+)...\n")
final_stats = stress_test()

print("\n✅ Stress test complete. Check results/stress_test.json for details.")

In [ ]:
import json

# Display results
with open("results/stress_test.json", "r") as f:
    data = json.load(f)

print("\n" + "="*70)
print("STRESS TEST SUMMARY")
print("="*70)

final = data["final_stats"]
print(f"\nFinal Capacity: {final['memory']['activation_ratio']*100:.1f}%")
print(f"Active Slots: {final['memory']['active_slots']}/{final['memory']['total_slots']}")
print(f"Rejection Rate (repetitive phase): {data['repetitive_rejection_rate']:.1%}")

# Show threshold progression
interactions = data["interactions"]
print(f"\nThreshold Progression:")
print(f"  Early (0-10): {interactions[0]['threshold']:.3f} - {interactions[9]['threshold']:.3f}")
print(f"  Mid (20-30): {interactions[20]['threshold']:.3f} - {interactions[29]['threshold']:.3f}")
print(f"  Late (40-50): {interactions[40]['threshold']:.3f} - {interactions[49]['threshold']:.3f}")

print(f"\nRepetitive Phase Rejection:")
for i in range(40, 50):
    r = interactions[i]
    status = "❌ REJECTED" if not r['stored'] else "✅ STORED"
    print(f"  {r['prompt']:20s} | Threshold: {r['threshold']:.3f} | {status}")